# AdaBoost

In [2]:
import scipy.io
import numpy as np
import sklearn
from sklearn.feature_extraction import image
import matplotlib.pyplot as plt
from helper_functions import *

## Dataset - Data

In [3]:
data = scipy.io.loadmat('Data/data.mat')
faces = np.array(data['face'])
#number of classes 
M = 2
N = 400 # 3 per class (ignoring 3rd image in each class, done in next cell)

In [4]:
faces.shape

(24, 21, 600)

   ### Split Data

In [5]:

#select images 2 and 3 from each subject for training
faces_flat = faces.reshape(-1,faces.shape[2])
#ignore image 3 (illumination) of all classes (creates a dataset of 400 samples)
idx = []
for n in range(600):
    if n%3 == 0 or n%3 == 1:
        idx = idx + [n]
faces_flat = faces_flat[:,idx]
train_idx =[]
test_idx = []
print(faces_flat.shape)
#create training set (75% = 300)
training_faces = faces_flat[:,:300]
test_faces = faces_flat[:,300:400]

# #rn test and training set have neutral face and expression alternating
#Create label Arrays
training_labels = [-1,1]*150
test_labels = [-1,1]*50

# training_faces = np.vstack(([-1,1]*150, training_faces))
# test_faces = np.vstack(([-1,1]*50, test_faces))

#datasets have alternating neutral and expression pictures
#rearrange saw that all the neutral come first then expression
training_faces = np.hstack((training_faces[:,::2], training_faces[:,1::2])) #0 -149 neutral, 150 - 299 expression
test_faces = np.hstack((test_faces[:,::2], test_faces[:,1::2])) #0-49 neutral, 50 - 99 expression
print(training_faces.shape, test_faces.shape)

(504, 400)
(504, 300) (504, 100)


### Perform PCA

In [6]:
#reduce dimensions from 504 to 1 (only 2 classes, so 1 dimension is the only possibility).
A, train_mean = PCA(training_faces, 3)
X_PCA = A @ (training_faces - train_mean)
X_test_PCA = A @ (test_faces - train_mean)
print(X_PCA.shape)


(3, 300)


### Perform MDA

In [7]:
#We have two classes here
A,class_means,sigma_w = MDA(X_PCA, 1, 2)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA

X_train = X_PCA.real
X_test = X_test_PCA.real
y_train = np.concatenate((np.zeros((1,150)), np.ones((1,150))), axis = 1)

### Defining stump as a class  

In [13]:
class DecisionStump:
    def __init__(self):
        self.k = None#idx
        self.theta = None#threshold
        self.s = 1#polarity

    def predict(self, X):
        return np.sign(self.s * (X[:, self.k] - self.theta))


### Finding good classifiers for AdaBoost


In [8]:
l, N_train = X_train.shape
T = 10 #no of rounds


#T stumps for adaboost
stumps = np.array([DecisionStump(i) for i in range(T)], dtype=object)

err = np.zeros(T)

a = np.zeros(T) #parameters (vote weightage of each weak learner)

#initialize weight w for priors
w = np.ones(T)

for t in range(T):
    #priors
    P = w/np.sum(w) # N_train x 1
    #returns feature index for stump, theta for threshold and error value err and polarity
    pol[t]*k[t],theta[t],err[t] = weak_classifier(X_train,w)
    a[t] = 0.5 * np.log((1-err[t])/err[t])
    #update weights
    phi = np.sign(pol[t]*x_train[k[t]] - theta[t])
    w = w * exp(- a[t])

NameError: name 'weak_classifier' is not defined

In [12]:
w = np.ones(N_train)
w-3

array([-2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.,
       -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2